# 🚀 04. LangGraph 기반 멀티 에이전트 및 구조화 응답

이 노트북은 LangGraph를 기반으로 한 다자간 에이전트(Multi-agent) 상태 기계 오케스트레이션과 체크포인터 복원, 그리고 LangChain `with_structured_output` 기법의 필수 타입 가드 검증 구조에 대해 학습합니다.

### ⚙️ 필요한 라이브러리 및 모듈 임포트
LangGraph 핵심 모듈과 Pydantic DTO 선언을 위한 필수 클래스들을 임포트합니다.

In [ ]:
import asyncio
from typing import Annotated, Optional, Any
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from api.database.config.psycopg_pool import psycopg_pool

## 📌 1단계. 구조화된 응답(Structured Output) 및 타입 가드
최종 응답을 DTO 구조로 강제 수신하기 위해 Pydantic 모델을 정의하고, 구조화 모델이 정상적으로 리턴되었는지 검사하는 명시적 타입 가드(Type Guard) 패턴을 구축합니다.

In [ ]:
class PaperReference(BaseModel):
    arxiv_id: str = Field(description="arXiv ID of the cited paper")
    title: str = Field(description="Title of the paper")
    relevance: str = Field(description="Relevance summary")

class AgentStructuredAnswer(BaseModel):
    explanation: str = Field(description="Markdown text explanation answering the question")
    citations: list[PaperReference] = Field(default_factory=list, description="List of cited papers")

def validate_structured_response(output: Any) -> AgentStructuredAnswer:
    if not isinstance(output, AgentStructuredAnswer):
        raise TypeError(f"Expected AgentStructuredAnswer, got {type(output)}")
    return output

# 검증 테스트
dummy_answer = AgentStructuredAnswer(explanation="Test content", citations=[])
try:
    verified = validate_structured_response(dummy_answer)
    print("Validation Successful! Response verified.")
except TypeError as te:
    print("Validation Failed!:", te)

## 📌 2단계. LangGraph 상태 정의 및 Reducer 패턴
멀티턴 대화 진행 중에 여러 검색 도구(Tools)가 병렬로 수집한 논문 출처들을 중복 없이 합산하고 누적하는 커스텀 리듀서(Reducer) 및 상태 스키마를 구성합니다.

In [ ]:
def merge_sources(left: list[dict] | None, right: list[dict] | None) -> list[dict]:
    if not right:
        return left or []
    # 'clear' 액션 수신 시 히스토리 청소
    for item in right:
        if isinstance(item, dict) and item.get("action") == "clear":
            return [x for x in right if x.get("action") != "clear"]
            
    combined = (left or []) + right
    seen = set()
    unique = []
    for s in combined:
        doc_id = s.get("arxiv_id") or ""
        if doc_id not in seen:
            seen.add(doc_id)
            unique.append(s)
    return unique

class MultiAgentState(BaseModel):
    messages: list[BaseMessage] = Field(default_factory=list)
    sources: Annotated[list[dict], merge_sources] = Field(default_factory=list)
    final_answer: Optional[AgentStructuredAnswer] = None

## 📌 3단계. 다자간 에이전트 노드(Nodes) 및 엣지(Edges) 정의
LangGraph에서 상태 데이터를 변경하고 실행 제어를 이어받는 각 노드(bio_agent, cs_agent, synthesizer) 함수를 정의합니다.

In [ ]:
async def bio_agent_node(state: MultiAgentState) -> dict:
    print("[Node] bio_agent running...")
    mock_source = {"arxiv_id": "2104.99999", "title": "Bio-inspired GNNs", "type": "bio"}
    return {
        "sources": [mock_source],
        "messages": [AIMessage(content="Bio-agent complete.")]
    }

async def cs_agent_node(state: MultiAgentState) -> dict:
    print("[Node] cs_agent running...")
    mock_source = {"arxiv_id": "2006.11367", "title": "Attention is all you need", "type": "cs"}
    return {
        "sources": [mock_source],
        "messages": [AIMessage(content="CS-agent complete.")]
    }

async def synthesizer_node(state: MultiAgentState) -> dict:
    print("[Node] synthesizer running...")
    sources = state.sources
    citations = [
        PaperReference(
            arxiv_id=s.get("arxiv_id", ""),
            title=s.get("title", ""),
            relevance=f"Found via {s.get('type')} search"
        ) for s in sources
    ]
    
    answer = AgentStructuredAnswer(
        explanation="## Synthesized Analysis\n\nCross-disciplinary study proves GNN works with attention.",
        citations=citations
    )
    # 검증 적용
    verified = validate_structured_response(answer)
    return {
        "final_answer": verified,
        "messages": [AIMessage(content="Synthesis complete.")]
    }

## 📌 4단계. 에이전트 컴파일 및 체크포인터 복원 멀티턴 대화 실습
상태 그래프를 구축하여 각 노드를 연결하고, `AsyncPostgresSaver` 체크포인터를 탑재해 스레드 ID별 대화 기록이 DB에서 로드 및 복원되는 과정을 확인합니다. (DB 오프라인 시 `MemorySaver`로 자동 폴백합니다.)

In [ ]:
# 그래프 생성
workflow = StateGraph(MultiAgentState)
workflow.add_node("bio_agent", bio_agent_node)
workflow.add_node("cs_agent", cs_agent_node)
workflow.add_node("synthesizer", synthesizer_node)

workflow.add_edge(START, "bio_agent")
workflow.add_edge("bio_agent", "cs_agent")
workflow.add_edge("cs_agent", "synthesizer")
workflow.add_edge("synthesizer", END)

# 체크포인터 초기화
checkpointer = None
try:
    if psycopg_pool.closed:
        await psycopg_pool.open()
    checkpointer = AsyncPostgresSaver(psycopg_pool)
    await checkpointer.setup()
    print("PostgreSQL Checkpointer initialized successfully! 🎉")
except Exception as e:
    print(f"PostgreSQL connection failed ({e}). Falling back to MemorySaver.")
    checkpointer = MemorySaver()

# 컴파일
app = workflow.compile(checkpointer=checkpointer)
config = {"configurable": {"thread_id": "notebook_agent_thread_123"}}

# 1회차 실행
print("\n--- Running Graph (1st Turn) ---")
result = await app.ainvoke(
    {"messages": [HumanMessage(content="Tell me about GNNs")], "sources": [{"action": "clear"}]},
    config=config
)

print(f"Final Answer Explanation: {result['final_answer'].explanation}")

# 2회차: 체크포인터로부터 복원
print("\n--- Verifying Checkpoint Restoration ---")
state_data = await app.aget_state(config)
print(f"Restored Messages Count: {len(state_data.values.get('messages', []))}")
print(f"Restored Sources List: {state_data.values.get('sources')}")

# psycopg 풀 닫기
try:
    await psycopg_pool.close()
except Exception:
    pass